In [2]:
import json

# The combinations of characters we’ll try appending
# when we detect invalid JSON. These will be tried in order.
FIX_COMBOS = [
    "",       # 1) No change, just try parsing as is
    "}",      # 2) Add one brace
    "\"",     # 3) Add a quote
    "\"}",    # 4) Add quote then brace
    "}\"",    # 5) Add brace then quote
    "\"}\""   # 6) Add quote, brace, quote
]

def try_fix_json(json_string):
    """
    Tries parsing 'json_string' with various appended fix combos.
    Returns the first successfully parsed object or None if none worked.
    """
    for combo in FIX_COMBOS:
        candidate = json_string + combo
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    return None  # Unable to parse with any appended combos

def fix_line_outer_and_review(line):
    """
    1) Attempt to parse the entire line as JSON (outer).
    2) If that fails, try appending possible combos (} and ") until it works.
    3) Then parse the 'review' key as JSON (inner). If that fails,
       append combos again.
    4) If both parse, re-embed the fixed 'review' as a proper string into
       the outer object, and return a valid JSON string for the entire line.
    5) Return None if we can’t fix it.
    """
    line = line.rstrip("\n")
    if not line.strip():
        return None

    # 1) Fix the outer JSON
    outer_data = try_fix_json(line)
    if outer_data is None:
        return None  # Can't fix outer JSON

    # 2) We have a dict; now fix the 'review' field if it exists
    if "review" not in outer_data:
        # If there's no 'review' field, just re-dump
        return json.dumps(outer_data)

    review_str = outer_data["review"]
    if not isinstance(review_str, str):
        # If 'review' isn't a string, nothing to fix
        return json.dumps(outer_data)

    # 3) Try to parse the review_str as JSON
    fixed_inner_data = try_fix_json(review_str)
    if fixed_inner_data is None:
        # Could not parse even after appending combos
        return None

    # 4) Re-embed the fixed 'review' data as a string
    outer_data["review"] = json.dumps(fixed_inner_data, ensure_ascii=False)

    # 5) Return a valid JSON line
    return json.dumps(outer_data, ensure_ascii=False)


def fix_missing_quotes_and_braces(input_file, output_file):
    """
    Reads each line of 'input_file', attempts to repair truncated JSON 
    by adding up to a couple of braces/quotes at the end, then writes
    out the fixed (or original if no fix needed) lines to 'output_file'.
    Skips lines we cannot fix.
    """
    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        
        for i, line in enumerate(fin, start=1):
            original_line = line.rstrip("\n")
            fixed_line = fix_line_outer_and_review(original_line)
            if fixed_line is None:
                print(f"Line {i}: Could not fix JSON by appending braces/quotes. Skipping.\n"
                      f"  Original line: {original_line}\n")
                continue

            fout.write(fixed_line + "\n")


if __name__ == "__main__":
    input_path = "results/hundred_shot_test_data_2024_abstract_prompts.jsonl"
    output_path = "results/hundred_shot_test_data_2024_abstract_prompts_fixed.jsonl"
    fix_missing_quotes_and_braces(input_path, output_path)
    print("Done attempting to fix truncated JSON lines.")


Line 18: Could not fix JSON by appending braces/quotes. Skipping.
  Original line: {"paper_id": "bAMPOUF227", "review": "{\"Soundness\": 2, \"Presentation\": 3, \"Contribution\": 2, \"Rating\": 3, \"Confidence\": 4, \"Strengths\": \"1. The paper is well-written and easy to follow.\\n2. The proposed method is simple yet effective, and it is easy to implement.\\n3. The paper is well-motivated, and the authors have conducted extensive experiments to evaluate their proposed method.\", \"Weaknesses\": \"1. The proposed method is not novel, and it is not clear how it is different from the existing methods. The authors should provide a more detailed comparison with the existing methods.\\n2. The paper does not provide a clear hypothesis about why the proposed method can improve the performance of the LLMs. It is not clear why the proposed method can improve the performance of the LLMs, and the authors should provide a more detailed explanation.\\n3. The paper does not provide a clear analysis

In [1]:
def parse_reviews(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
        for i, line in enumerate(f_in, start=1):
            line = line.strip()
            # Skip empty lines
            if not line:
                continue
            try:
                data = json.loads(line)  # Parse the main JSON line
                paper_id = data.get('paper_id', None)
                review_str = data.get('review', "")

                try:
                    review_data = json.loads(review_str)  # Parse the 'review' field as JSON
                except json.JSONDecodeError as e:
                    # More detailed logging for JSON decode problems
                    print(f"Line {i}, paper_id={paper_id}: JSON Decode Error: {e}")
                    print(f"  Full line contents: {line}")
                    print(f"  Review field (raw) : {review_str}")
                    print("Skipping this line.")
                    continue
                except Exception as e:
                    # Catch any other exceptions when decoding review
                    print(f"Line {i}, paper_id={paper_id}: Unknown error decoding 'review': {e}")
                    print(f"  Full line contents: {line}")
                    print(f"  Review field (raw) : {review_str}")
                    print("Skipping this line.")
                    continue

                # Construct the new dictionary to write out
                parsed = {
                    "paper_id": paper_id,
                    "Soundness": review_data.get("Soundness", None),
                    "Presentation": review_data.get("Presentation", None),
                    "Contribution": review_data.get("Contribution", None),
                    "Rating": review_data.get("Rating", None),
                    "Confidence": review_data.get("Confidence", None),
                    "Strengths": review_data.get("Strengths", ""),
                    "Weaknesses": review_data.get("Weaknesses", "")
                }

                # Write out as a JSON line
                f_out.write(json.dumps(parsed, ensure_ascii=False) + "\n")

            except json.JSONDecodeError as e:
                print(f"Line {i}: JSON Decode Error reading outer JSON object: {e}")
                print(f"  Full line contents: {line}")
                print("Skipping this line.")
            except Exception as e:
                print(f"Line {i}, paper_id={paper_id}: Unexpected error: {e}")
                print(f"  Full line contents: {line}")
                print("Skipping this line.")

if __name__ == "__main__":
    input_path = "results/hundred_shot_test_data_2024_abstract_prompts_fixed.jsonl"
    output_path = "results/hundred_shot_test_data_2024_abstract_prompts_parsed.jsonl"
    parse_reviews(input_path, output_path)
    print("Parsing complete.")


Line 18: Could not fix JSON by appending braces/quotes. Skipping.
  Original line: {"paper_id": "bAMPOUF227", "review": "{\"Soundness\": 2, \"Presentation\": 3, \"Contribution\": 2, \"Rating\": 3, \"Confidence\": 4, \"Strengths\": \"1. The paper is well-written and easy to follow.\\n2. The proposed method is simple yet effective, and it is easy to implement.\\n3. The paper is well-motivated, and the authors have conducted extensive experiments to evaluate their proposed method.\", \"Weaknesses\": \"1. The proposed method is not novel, and it is not clear how it is different from the existing methods. The authors should provide a more detailed comparison with the existing methods.\\n2. The paper does not provide a clear hypothesis about why the proposed method can improve the performance of the LLMs. It is not clear why the proposed method can improve the performance of the LLMs, and the authors should provide a more detailed explanation.\\n3. The paper does not provide a clear analysis